# OpenNext CMS — Stage 1 + Stage 2 pipeline (GPU)
Trained Qwen2.5-1.5B LoRA turns your brief into a LayoutPlan, then OpenRouter fills the copy.
**Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
# 1) Install deps (Colab already has a CUDA build of torch)
!pip -q install "transformers>=4.44" "peft>=0.11" accelerate safetensors requests json-repair
import torch
print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set Runtime > Change runtime type > T4 GPU")

In [ ]:
# 2) Get the LoRA adapter into Colab
# --- Option A (default): upload a ZIP of your adapter folder ---
# On your PC, zip the folder training/layout-planner/outputs/qwen-layout-planner
# (it must contain adapter_config.json + adapter_model.safetensors + tokenizer files),
# then run this cell and pick that .zip.
import os, glob, zipfile
from google.colab import files
up = files.upload()                      # choose qwen-layout-planner.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall("/content/_adapter")
cfg = glob.glob("/content/_adapter/**/adapter_config.json", recursive=True)[0]
ADAPTER_DIR = os.path.dirname(cfg)
print("ADAPTER_DIR =", ADAPTER_DIR)

# --- Option B (instead of A): pull the adapter from the HF Hub ---
# ADAPTER_DIR = "your-hf-username/qwen-layout-planner"

In [ ]:
# 3) Config — SYSTEM prompt is byte-identical to inference.py; ALIASES/FILLER match pipeline.py
import json, re, os

BASE_MODEL   = "Qwen/Qwen2.5-1.5B-Instruct"
INPUT_PREFIX = "plan layout: "

SYSTEM = (
    "You are a website STRUCTURE planner for OpenNext CMS. "
    "Given a brief, return ONE minified JSON LayoutPlan and nothing else. "
    "Shape: {meta:{business,style,websiteType,pageName,location,complexity},"
    "page:{theme:{primaryColor,secondaryColor,backgroundColor,textColor,fontFamily,borderRadius},"
    "blocks:[{id,type,order,styles,content}]}}. "
    "Rules: blocks[0] is a navigation bar and the last block is a footer (exactly one each); "
    "exactly one hero, placed before any feature/testimonial/pricing/content/stats section; "
    "every block.content is {} (NO copy); ids are unique kebab-case; order is 0..n-1; "
    "colours are hex and section backgrounds vary; "
    "complexity simple=4-6 blocks, standard=7-10, rich=11-15."
)

# planner block-type  ->  v2 filler type (from schema/allowed-block-types.json)
ALIASES = {
    "nav-bar": "navigation", "hero": "hero", "hero-main": "heroCentered", "hero-centered": "heroCentered",
    "feature-trio": "featureGrid", "feature-vertical": "featureCards", "feature-horizontal": "featureCards",
    "feature-boxed": "featureCards", "feature-zigzag": "featureCards", "feature-side-image": "featureCards",
    "feature-checklist": "featureGrid", "feature-list": "featureGrid", "content-features": "featureGrid",
    "content-gallery": "gallery", "content-split": "featureCards", "content-trio": "featureGrid",
    "content-icons": "featureGrid", "content-categories": "gallery", "content-detail": "featureCards",
    "testimonial-main": "testimonials", "testimonial-single": "testimonials",
    "testimonial-single-large": "testimonials", "stats": "statistics", "statistics-main": "statistics",
    "statistics-side-image": "statistics", "statistics-boxed": "statistics", "ecommerce-grid": "gallery",
    "ecommerce-detail": "gallery", "pricing": "pricing", "cta": "cta", "contact": "contact",
    "contact-simple": "contact", "footer": "footer",
}

FILLER_SYSTEM = (
    "You are the Stage-2 CONTENT filler for OpenNext CMS. You are given a FIXED page "
    "structure: a list of blocks, each with an id, a type, a styles object, and an empty "
    "data object. Fill ONLY the copy.\n\n"
    "RULES:\n"
    "- Return ONE JSON object: {\"meta\":{...},\"theme\":{...},\"blocks\":[{id,type,styles,data}]}.\n"
    "- Keep EVERY block's id, type, styles and ORDER exactly as given. Do NOT add, remove, "
    "reorder, or retype blocks.\n"
    "- Populate each block's `data` with realistic, niche-specific copy for the brief "
    "(headlines <=10 words, subheadlines <=30 words, feature text <=20 words). No lorem ipsum.\n"
    "- Use the data shape appropriate to each v2 type: navigation{logo,links,cta}, "
    "heroCentered/hero{eyebrow,headline,subheadline,cta_primary,media}, "
    "featureGrid/featureCards{headline,features|cards:[{title,description,icon}]}, "
    "testimonials{headline,items:[{quote,author_name,author_title}]}, "
    "statistics{headline,stats:[{value,label}]}, pricing{headline,plans:[{name,price,features}]}, "
    "cta{headline,subheadline,cta_primary}, contact{headline,email,phone,address}, "
    "gallery{headline,items:[{src,caption}]}, footer{logo,columns:[{title,links}],copyright}.\n"
    "- Every media/avatar/logo field gets a real Unsplash/Pexels URL matching the niche.\n"
    "Return JSON only - no markdown, no prose."
)

def parse_plan_json(raw):
    """fence-strip -> json.loads -> brace-slice -> json_repair."""
    if not raw:
        return None
    s = raw.strip()
    m = re.search(r"```(?:json)?\s*(.*?)```", s, re.S)
    if m:
        s = m.group(1).strip()
    try:
        return json.loads(s)
    except Exception:
        pass
    i, j = s.find("{"), s.rfind("}")
    if i != -1 and j != -1 and j > i:
        frag = s[i:j + 1]
        try:
            return json.loads(frag)
        except Exception:
            try:
                from json_repair import repair_json
                return json.loads(repair_json(frag))
            except Exception:
                return None
    return None

NAV, HERO, FOOT = {"nav-bar"}, {"hero", "hero-main", "hero-centered"}, {"footer"}
def validate(plan):
    if not isinstance(plan, dict):
        return False, "not a dict"
    page = plan.get("page")
    if not isinstance(page, dict) or not isinstance(page.get("blocks"), list) or not page["blocks"]:
        return False, "no page.blocks"
    t = [b.get("type") for b in page["blocks"] if isinstance(b, dict)]
    if t[0] not in NAV:      return False, "first block is not a nav-bar"
    if t[-1] not in FOOT:    return False, "last block is not a footer"
    if sum(x in HERO for x in t) != 1: return False, "need exactly one hero"
    return True, "ok"

print("config loaded")

In [ ]:
# 4) Load base Qwen + LoRA adapter on the GPU (fp16)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tok_src = ADAPTER_DIR if os.path.exists(os.path.join(ADAPTER_DIR, "tokenizer_config.json")) else BASE_MODEL
tok = AutoTokenizer.from_pretrained(tok_src)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()
print("model ready on", model.device)

In [ ]:
# 5) Pipeline functions (Stage 1 = model, Stage 2 = OpenRouter fill)
import requests

def generate(brief, max_new_tokens=768, do_sample=False, temperature=1.0):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": INPUT_PREFIX + brief.strip()}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(model.device)
    kw = dict(max_new_tokens=max_new_tokens, do_sample=do_sample, pad_token_id=tok.pad_token_id)
    if do_sample:
        kw["temperature"] = temperature
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

def stage1(brief):
    plan = parse_plan_json(generate(brief, do_sample=False))          # greedy
    if plan and validate(plan)[0]:
        return plan, "model"
    plan2 = parse_plan_json(generate(brief, do_sample=True, temperature=0.7))  # genuine sampled retry
    if plan2 and validate(plan2)[0]:
        return plan2, "model-retry"
    return (plan or plan2), "model-unvalidated"   # best effort

def to_v2_skeleton(plan):
    page = plan.get("page", {}); meta = plan.get("meta", {}); th = page.get("theme", {})
    blocks = [{"id": b.get("id"), "type": ALIASES.get(b.get("type"), b.get("type")),
               "styles": b.get("styles", {}), "data": {}} for b in page.get("blocks", [])]
    return {"meta": {k: meta.get(k) for k in ("business", "websiteType", "style", "pageName", "location")},
            "theme": {"primary_color": th.get("primaryColor"), "secondary_color": th.get("secondaryColor"),
                      "background_color": th.get("backgroundColor"), "text_color": th.get("textColor"),
                      "font_body": th.get("fontFamily"), "border_radius": th.get("borderRadius")},
            "blocks": blocks}

def fill(skeleton, brief, model_id="openai/gpt-4o-mini"):
    msgs = [{"role": "system", "content": FILLER_SYSTEM},
            {"role": "user", "content": f"BRIEF: {brief}\n\nSTRUCTURE (fill each block's data, keep everything else):\n"
                                        + json.dumps(skeleton, ensure_ascii=False)}]
    r = requests.post("https://openrouter.ai/api/v1/chat/completions",
                      headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
                               "Content-Type": "application/json"},
                      json={"model": model_id, "temperature": 0.7, "max_tokens": 4000, "messages": msgs},
                      timeout=120)
    r.raise_for_status()
    return parse_plan_json(r.json()["choices"][0]["message"]["content"]) or skeleton

print("functions ready")

In [ ]:
# 6) RUN END-TO-END  —  your brief is the user input
import getpass, time

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key (sk-or-...): ")

brief = input("Describe the website you want: ").strip()

t0 = time.time()
plan, source = stage1(brief)
print(f"\nStage 1 [{source}] in {time.time()-t0:.1f}s | blocks:",
      [b["type"] for b in plan["page"]["blocks"]])
print("\n===== STAGE 1 - LAYOUT PLAN (structure only, model output) =====")
print(json.dumps(plan, indent=2, ensure_ascii=False))

skeleton = to_v2_skeleton(plan)
t1 = time.time()
filled = fill(skeleton, brief)
print(f"\nStage 2 fill in {time.time()-t1:.1f}s")
print("\n===== STAGE 2 - FILLED v2 PAGE JSON (copy added) =====")
print(json.dumps(filled, indent=2, ensure_ascii=False))

# save both stages to a downloadable file
with open("/content/page_both_stages.json", "w", encoding="utf-8") as f:
    json.dump({"brief": brief, "stage1_source": source,
               "stage1_layout_plan": plan, "stage2_filled_page": filled}, f, indent=2, ensure_ascii=False)
print("\nsaved -> /content/page_both_stages.json  (Files panel on the left to download)")